In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
customers = pd.read_csv(os.path.join(path, "customers.csv"))
geography = pd.read_csv(os.path.join(path, "geography.csv"))
inventory = pd.read_csv(os.path.join(path, "inventory.csv"))
order_items = pd.read_csv(os.path.join(path, "order_items.csv"))
orders = pd.read_csv(os.path.join(path, "orders.csv"))
payments = pd.read_csv(os.path.join(path, "payments.csv"))
products = pd.read_csv(os.path.join(path, "products.csv"))
promotions = pd.read_csv(os.path.join(path, "promotions.csv"))
returns = pd.read_csv(os.path.join(path, "returns.csv"))
reviews = pd.read_csv(os.path.join(path, "reviews.csv"))
sales = pd.read_csv(os.path.join(path, "sales.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))
shipments = pd.read_csv(os.path.join(path, "shipments.csv"))
web_traffic = pd.read_csv(os.path.join(path, "web_traffic.csv"))

print("Load xong tất cả DataFrame")

/tmp/ipykernel_3844/177485643.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(os.path.join(path, "order_items.csv"))


Load xong tất cả DataFrame


In [ ]:
# CHECK LẠI CÁC DATAFRAME
for name, df in {
    "customers": customers,
    "geography": geography,
    "inventory": inventory,
    "order_items": order_items,
    "orders": orders,
    "payments": payments,
    "products": products,
    "promotions": promotions,
    "returns": returns,
    "reviews": reviews,
    "sales": sales,
    "sample_submission": sample_submission,
    "shipments": shipments,
    "web_traffic": web_traffic
}.items():
    print(f"{name:<20} {df.shape}")

customers            (121930, 7)
geography            (39948, 4)
inventory            (60247, 17)
order_items          (714669, 7)
orders               (646945, 8)
payments             (646945, 4)
products             (2412, 8)
promotions           (50, 10)
returns              (39939, 7)
reviews              (113551, 7)
sales                (3833, 3)
sample_submission    (548, 3)
shipments            (566067, 4)
web_traffic          (3652, 7)


# PHẦN 1: CÂU HỎI TRẮC NGHIỆM

### Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [ ]:
# Q1
q1_df = orders.sort_values(["customer_id", "order_date", "order_id"]).copy()
q1_df["order_date"] = pd.to_datetime(q1_df["order_date"])
q1_df["prev_order_date"] = q1_df.groupby("customer_id")["order_date"].shift()
q1_df["inter_order_gap"] = (q1_df["order_date"] - q1_df["prev_order_date"]).dt.days

q1_result = q1_df.loc[q1_df["inter_order_gap"].notna(), "inter_order_gap"].median()
print(q1_result)

144.0


### Q2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price −cogs)/price?

In [ ]:
# Q2
q2_df = products.copy()
q2_df["gross_margin"] = (q2_df["price"] - q2_df["cogs"]) / q2_df["price"]

q2_result = q2_df.groupby("segment")["gross_margin"].mean().sort_values(ascending = False)
print(q2_result)
print(f"Phân khúc có tỷ suất lợi nhuận gộp tb cao nhất: { q2_result.idxmax()}")

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64
Phân khúc có tỷ suất lợi nhuận gộp tb cao nhất: Standard


### Q3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [ ]:
# Q3
q3_df = returns.merge(products[["product_id", "category"]], on="product_id")
q3_df = q3_df[q3_df["category"] == "Streetwear"]

q3_result = q3_df["return_reason"].value_counts()
print(q3_result)
print(f"Lý do trả hàng xuất hiện nhiều nhất: {q3_result.idxmax()} ")

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
Lý do trả hàng xuất hiện nhiều nhất: wrong_size 


### Q4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [ ]:
q4_df = web_traffic.copy()
q4_result = q4_df.groupby("traffic_source")["bounce_rate"].mean().sort_values()
print(q4_result)
print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: {q4_result.idxmin()} ")

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: email_campaign 


### Q5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [ ]:
# Q5
q5_result = order_items["promo_id"].notna().mean()*100
print(q5_result)

38.663493169565214


### Q6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [ ]:
# Q6
q6_orders = orders.groupby("customer_id")["order_id"].count().reset_index()
q6_orders.columns = ["customer_id", "order_count"]
q6_df = customers[customers["age_group"].notna()].merge(q6_orders, on="customer_id", how="left")
q6_df["order_count"] = q6_df["order_count"].fillna(0)

q6_result = q6_df.groupby("age_group")["order_count"].mean().sort_values(ascending=False)
print(q6_result)
print(f"Nhóm tuổi có số đơn hàng tb trên mỗi khách hàng cao nhất: {q6_result.idxmax()}")

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64
Nhóm tuổi có số đơn hàng tb trên mỗi khách hàng cao nhất: 55+


### Q7:  Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.cs?

In [ ]:
# Q7 - tính revenue từ order_items vì sales.csv không có region
q7_df = order_items.copy()
q7_df["revenue"] = q7_df["unit_price"] * q7_df["quantity"] - q7_df["discount_amount"]
q7_df = q7_df.merge(orders[["order_id", "zip"]], on="order_id")
q7_df = q7_df.merge(geography[["zip", "region"]], on="zip")

q7_result = q7_df.groupby("region")["revenue"].sum().sort_values(ascending=False)
print(q7_result)
print(f"Region tạo ra tổng doanh thu cao: {q7_result.idxmax()}")

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: revenue, dtype: float64
Region tạo ra tổng doanh thu cao: East


### Q8: Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [ ]:
# Q8
q8_df = orders[orders["order_status"] == "cancelled"]
q8_result = q8_df["payment_method"].value_counts().sort_values(ascending=False)
print(q8_result)
print(f"Phương thức thanh toán được sử dụng nhiều nhất: {q8_result.idxmax()}")

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Phương thức thanh toán được sử dụng nhiều nhất: credit_card


### Q9: Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [ ]:
# Q9
q9_items = order_items.merge(products[["product_id", "size"]], on="product_id")
q9_returns = returns.merge(products[["product_id", "size"]], on="product_id")

q9_items_count = q9_items.groupby("size")["order_id"].count()
q9_returns_count = q9_returns.groupby("size")["return_id"].count()

q9_result = (q9_returns_count / q9_items_count).sort_values(ascending=False)
print(q9_result)
print(f"Kích thước có tỷ lệ trả hàng cao nhất: {q9_result.idxmax()}")

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64
Kích thước có tỷ lệ trả hàng cao nhất: S


### Q10: Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [ ]:
# Q10
q10_result = payments.groupby("installments")["payment_value"].mean().sort_values(ascending=False)
print(q10_result)
print(f"Kế hoạch trả góp có giá trị thanh toán tb trên mỗi đơn hàng cao nhất: {q10_result.idxmax()}")

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
Kế hoạch trả góp có giá trị thanh toán tb trên mỗi đơn hàng cao nhất: 6
